# FMECAF: Full Working Implementation Notebook
**Reliability-Aware Federated Multimodal Learning with Evidence-Grounded RAG for Cardiovascular Assessment**

This notebook implements an executable research prototype of the manuscript's pipeline:
1. Patient-matched ECG, chest radiograph, and laboratory data loading.
2. Patient-disjoint train/validation/test partitions and five simulated federated clients.
3. Training-only preprocessing and explicit modality-availability masks.
4. RF-MambaFusion-style modality encoders, reliability estimation, masked cross-modal attention, and reliability-aware fusion.
5. Simulated federated training with branch-specific reliability-weighted aggregation.
6. Validation temperature scaling, test metrics, missing-modality evaluation, and plots.
7. PC-EvidenceRAG-style retrieval, source provenance, claim-level checks, and abstention.

**Important implementation notes**
- The manuscript specifies Symile-MIMIC v1.0.0, whose access requires PhysioNet credentials and a data-use agreement. Download/access it separately and point this notebook to a prepared manifest.
- The notebook does not fabricate clinical records or results. It can run a clearly labelled synthetic smoke test if `USE_SYNTHETIC_DEMO=True`; synthetic results are not clinical evidence.
- To remain runnable on common Colab GPUs/CPUs, the default encoders are compact PyTorch encoders. They are architecture-faithful prototypes, **not** pretrained Mamba-2, DINOv3, MedCPT, or MedGemma. Optional integrations are left as replaceable components.
- This is research code, not a medical device or diagnostic tool. Cardiomegaly is a radiographic finding, not a complete cardiovascular diagnosis.

## 1. Install dependencies
Run once in Colab/Jupyter. Restart the runtime only if the environment asks you to.

In [ ]:
%pip -q install numpy pandas scikit-learn matplotlib pillow tqdm

## 2. Imports, reproducibility, and configuration

In [ ]:
import os, json, math, time, copy, random, warnings, hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, brier_score_loss, log_loss,
    matthews_corrcoef, classification_report, precision_recall_curve, roc_curve
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Set this False for your real prepared cohort.
USE_SYNTHETIC_DEMO = True

# Real-data mode expects a manifest CSV with at least:
# patient_id, cardiomegaly, and one or more of:
# ecg_path, image_path, plus laboratory columns prefixed lab_
MANIFEST_CSV = "/content/drive/MyDrive/FMECAF/manifest.csv"
EVIDENCE_CSV = "/content/drive/MyDrive/FMECAF/medical_evidence.csv"  # optional columns: text,title,url,date
IMAGE_SIZE = 224
ECG_LENGTH = 5000                  # 10 sec * 500 Hz, flattened/interpolated for compact prototype
BATCH_SIZE = 16
NUM_CLIENTS = 5
FED_ROUNDS = 5                    # raise to manuscript target after confirming runtime
LOCAL_EPOCHS = 1                  # manuscript proposes 2 epochs/round
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
LATENT_DIM = 128
DROPOUT = 0.1
MISSINGNESS_RATES = [0.0, 0.1, 0.3, 0.5]

## 3. Data input contract and synthetic smoke-test cohort
In real-data mode, create a patient-level/encounter-level manifest. Each row must refer to one matched encounter; do not join modalities across unrelated encounters. ECG files may be `.npy` arrays (shape `[12, time]` or `[time, 12]`), images should be readable image files, and labs are numeric columns named `lab_*`. The label must be binary: 1 = cardiomegaly-positive, 0 = negative. Exclude uncertain/missing labels before running the notebook.

If the dataset release has a different layout, export its official patient/admission associations to this manifest rather than guessing matches.

In [ ]:
def make_synthetic_manifest(n=600, seed=SEED):
    """Clearly synthetic data for testing pipeline mechanics only."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        patient = f"SYN_P{i:05d}"
        y = int(rng.random() < 0.28)
        labs = rng.normal(0, 1, 12)
        labs[:3] += y * np.array([0.7, 0.5, -0.4])
        rows.append({
            "patient_id": patient,
            "encounter_id": f"SYN_E{i:05d}",
            "cardiomegaly": y,
            "ecg_path": "",
            "image_path": "",
            **{f"lab_{j:02d}": float(v) for j, v in enumerate(labs)}
        })
    return pd.DataFrame(rows)

if USE_SYNTHETIC_DEMO:
    df = make_synthetic_manifest()
    print("WARNING: synthetic smoke-test data only; not clinical results.")
else:
    if not os.path.exists(MANIFEST_CSV):
        raise FileNotFoundError(
            f"Manifest not found: {MANIFEST_CSV}. Set MANIFEST_CSV to your prepared Symile-MIMIC manifest."
        )
    df = pd.read_csv(MANIFEST_CSV)
    required = {"patient_id", "cardiomegaly"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Manifest is missing required columns: {missing}")
    df = df[df["cardiomegaly"].isin([0, 1])].copy()
    df["cardiomegaly"] = df["cardiomegaly"].astype(int)

if "encounter_id" not in df.columns:
    df["encounter_id"] = [f"enc_{i}" for i in range(len(df))]
lab_cols = [c for c in df.columns if c.startswith("lab_")]
if not lab_cols:
    raise ValueError("No lab columns found. Name numeric laboratory features with prefix 'lab_'.")
if df["patient_id"].isna().any():
    raise ValueError("patient_id contains missing values; resolve patient identity before partitioning.")
df = df.reset_index(drop=True)
print("Manifest shape:", df.shape)
print("Patients:", df.patient_id.nunique(), "| encounters:", len(df))
print("Label distribution:\n", df.cardiomegaly.value_counts(dropna=False).sort_index())
display(df.head())

## 4. Visualize label balance and modality availability before preprocessing

In [ ]:
availability_cols = []
for col, name in [("ecg_path", "ECG"), ("image_path", "Radiograph")]:
    if col in df.columns:
        avail = df[col].fillna("").astype(str).str.len().gt(0)
        if USE_SYNTHETIC_DEMO:
            avail[:] = True
        df[f"has_{name.lower()}"] = avail
        availability_cols.append(f"has_{name.lower()}")
df["has_labs"] = df[lab_cols].notna().any(axis=1)
availability_cols.append("has_labs")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["cardiomegaly"].value_counts().sort_index().plot(kind="bar", ax=ax[0])
ax[0].set_title("Label distribution before preprocessing"); ax[0].set_xlabel("Cardiomegaly label"); ax[0].set_ylabel("Encounters")
df[availability_cols].mean().plot(kind="bar", ax=ax[1])
ax[1].set_title("Modality availability before preprocessing"); ax[1].set_ylabel("Available fraction"); ax[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

## 5. Patient-disjoint train/validation/test split
The split is performed by patient ID. Every encounter from a patient stays in one partition. This prevents leakage between partitions. For an already partitioned official release, preserve its released split instead of creating a new one.

In [ ]:
def group_train_val_test(frame, group_col="patient_id", label_col="cardiomegaly",
                         test_size=0.15, val_size=0.15, seed=SEED):
    # First hold out test patients, then validation patients from the remaining patients.
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(frame, frame[label_col], groups=frame[group_col]))
    trainval = frame.iloc[trainval_idx].copy()
    test = frame.iloc[test_idx].copy()
    val_fraction_relative = val_size / (1.0 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_fraction_relative, random_state=seed + 1)
    tr_idx, val_idx = next(gss2.split(trainval, trainval[label_col], groups=trainval[group_col]))
    train = trainval.iloc[tr_idx].copy()
    val = trainval.iloc[val_idx].copy()
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

train_df, val_df, test_df = group_train_val_test(df)
sets = {"train": train_df, "validation": val_df, "test": test_df}
for name, part in sets.items():
    print(f"{name:10s}: rows={len(part):4d}, patients={part.patient_id.nunique():4d}, labels={part.cardiomegaly.value_counts().to_dict()}")
assert set(train_df.patient_id).isdisjoint(val_df.patient_id)
assert set(train_df.patient_id).isdisjoint(test_df.patient_id)
assert set(val_df.patient_id).isdisjoint(test_df.patient_id)

## 6. Training-only lab preprocessing
Imputation and scaling are fit on training rows only. Missingness indicators are retained. Validation and test data are transformed using the frozen training-derived parameters.

In [ ]:
lab_imputer = SimpleImputer(strategy="median", add_indicator=False)
lab_scaler = StandardScaler()

X_train_lab_raw = train_df[lab_cols].apply(pd.to_numeric, errors="coerce")
X_val_lab_raw = val_df[lab_cols].apply(pd.to_numeric, errors="coerce")
X_test_lab_raw = test_df[lab_cols].apply(pd.to_numeric, errors="coerce")

# Handle all-missing training columns safely by replacing them with zero before fitting.
all_missing = X_train_lab_raw.columns[X_train_lab_raw.isna().all()].tolist()
for c in all_missing:
    X_train_lab_raw[c] = 0.0
    X_val_lab_raw[c] = X_val_lab_raw[c].fillna(0.0)
    X_test_lab_raw[c] = X_test_lab_raw[c].fillna(0.0)

lab_imputer.fit(X_train_lab_raw)
Xtr_imp = lab_imputer.transform(X_train_lab_raw)
Xva_imp = lab_imputer.transform(X_val_lab_raw)
Xte_imp = lab_imputer.transform(X_test_lab_raw)
lab_scaler.fit(Xtr_imp)
Xtr_lab = lab_scaler.transform(Xtr_imp).astype(np.float32)
Xva_lab = lab_scaler.transform(Xva_imp).astype(np.float32)
Xte_lab = lab_scaler.transform(Xte_imp).astype(np.float32)

Mtr_lab = X_train_lab_raw.notna().to_numpy(dtype=np.float32)
Mva_lab = X_val_lab_raw.notna().to_numpy(dtype=np.float32)
Mte_lab = X_test_lab_raw.notna().to_numpy(dtype=np.float32)

print("Lab arrays:", Xtr_lab.shape, Xva_lab.shape, Xte_lab.shape)
print("Training-only imputer/scaler fitted.")

## 7. ECG and radiograph loaders
Real mode reads only paths provided in the manifest. ECG `.npy` supports `[12,T]` or `[T,12]`; values are resampled to a fixed length and converted to a compact signal summary tensor. For the prototype, the ECG encoder uses a 1-D convolutional network and the image encoder uses a compact 2-D CNN. Replace these with pretrained Mamba-2/DINOv3 branches for the exact manuscript configuration if the weights and runtime are available.

In [ ]:
def load_ecg(path, length=ECG_LENGTH):
    if not path or not isinstance(path, str) or not os.path.isfile(path):
        return np.zeros((12, length), dtype=np.float32), 0.0
    try:
        arr = np.load(path, allow_pickle=False).astype(np.float32)
        if arr.ndim == 1:
            arr = np.tile(arr[None, :], (12, 1))
        if arr.ndim != 2:
            raise ValueError("ECG array must be 1D or 2D")
        if arr.shape[0] != 12 and arr.shape[1] == 12:
            arr = arr.T
        if arr.shape[0] != 12:
            raise ValueError(f"Expected 12 leads, got shape {arr.shape}")
        # Linear resampling per lead.
        old_x = np.linspace(0, 1, arr.shape[1])
        new_x = np.linspace(0, 1, length)
        arr = np.stack([np.interp(new_x, old_x, lead) for lead in arr]).astype(np.float32)
        # Per-record robust clipping and scaling; avoid fitting on validation/test.
        med = np.median(arr, axis=1, keepdims=True)
        q1 = np.percentile(arr, 25, axis=1, keepdims=True)
        q3 = np.percentile(arr, 75, axis=1, keepdims=True)
        arr = (arr - med) / np.maximum(q3 - q1, 1e-5)
        arr = np.clip(arr, -10, 10)
        quality = float(np.isfinite(arr).all() and np.std(arr) > 1e-6)
        return np.nan_to_num(arr), quality
    except Exception as e:
        warnings.warn(f"Could not load ECG '{path}': {e}")
        return np.zeros((12, length), dtype=np.float32), 0.0

def load_radiograph(path, size=IMAGE_SIZE):
    if not path or not isinstance(path, str) or not os.path.isfile(path):
        return np.zeros((1, size, size), dtype=np.float32), 0.0
    try:
        img = Image.open(path).convert("L").resize((size, size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        quality = float(np.isfinite(arr).all() and arr.std() > 1e-5)
        arr = (arr - arr.mean()) / max(arr.std(), 1e-5)
        return arr[None, ...].astype(np.float32), quality
    except Exception as e:
        warnings.warn(f"Could not load image '{path}': {e}")
        return np.zeros((1, size, size), dtype=np.float32), 0.0

def build_modality_arrays(frame, lab_array, lab_mask):
    n = len(frame)
    ecgs = np.zeros((n, 12, ECG_LENGTH), dtype=np.float32)
    images = np.zeros((n, 1, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
    quality = np.zeros((n, 3), dtype=np.float32)
    for i, (_, row) in enumerate(frame.iterrows()):
        if USE_SYNTHETIC_DEMO:
            # Deterministic synthetic modality tensors to smoke-test model plumbing.
            rng = np.random.default_rng(SEED + i)
            y = int(row["cardiomegaly"])
            ecgs[i] = rng.normal(0, 0.25, size=(12, ECG_LENGTH)).astype(np.float32)
            ecgs[i, :, :250] += y * 0.15
            images[i] = rng.normal(0, 0.25, size=(1, IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32)
            images[i, :, 70:150, 70:150] += y * 0.15
            quality[i, 0] = 1.0
            quality[i, 1] = 1.0
        else:
            ecg_path = str(row.get("ecg_path", "")) if pd.notna(row.get("ecg_path", "")) else ""
            img_path = str(row.get("image_path", "")) if pd.notna(row.get("image_path", "")) else ""
            ecgs[i], quality[i, 0] = load_ecg(ecg_path)
            images[i], quality[i, 1] = load_radiograph(img_path)
        quality[i, 2] = float(np.mean(lab_mask[i])) if lab_mask.shape[1] else 0.0
    return ecgs, images, lab_array, lab_mask.astype(np.float32), quality

Xtr_ecg, Xtr_img, Xtr_lab, Xtr_labmask, Qtr = build_modality_arrays(train_df, Xtr_lab, Mtr_lab)
Xva_ecg, Xva_img, Xva_lab, Xva_labmask, Qva = build_modality_arrays(val_df, Xva_lab, Mva_lab)
Xte_ecg, Xte_img, Xte_lab, Xte_labmask, Qte = build_modality_arrays(test_df, Xte_lab, Mte_lab)
y_train = train_df.cardiomegaly.to_numpy(dtype=np.float32)
y_val = val_df.cardiomegaly.to_numpy(dtype=np.float32)
y_test = test_df.cardiomegaly.to_numpy(dtype=np.float32)

print("ECG:", Xtr_ecg.shape, "| images:", Xtr_img.shape, "| labs:", Xtr_lab.shape)

## 8. Visualize preprocessed examples
The displayed synthetic examples are only for code validation. For real data, confirm that ECG traces and radiographs look clinically plausible before training.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(Xtr_ecg[0, 0, :1000]); ax[0].set_title("Preprocessed ECG lead 1"); ax[0].set_xlabel("Sample")
ax[1].imshow(Xtr_img[0, 0], cmap="gray"); ax[1].set_title("Preprocessed radiograph")
ax[2].bar(["observed lab fraction"], [float(Xtr_labmask[0].mean())]); ax[2].set_ylim(0, 1.05); ax[2].set_title("Laboratory completeness")
plt.tight_layout(); plt.show()

## 9. Dataset and dataloaders
Each example includes modality availability. In real mode, a missing or invalid file is represented by a zero tensor **plus a false availability mask**, not treated as a real observation.

In [ ]:
class FMECAFDataset(Dataset):
    def __init__(self, ecg, image, lab, labmask, quality, labels):
        self.ecg = torch.tensor(ecg, dtype=torch.float32)
        self.image = torch.tensor(image, dtype=torch.float32)
        self.lab = torch.tensor(lab, dtype=torch.float32)
        self.labmask = torch.tensor(labmask, dtype=torch.float32)
        self.quality = torch.tensor(quality, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        q = self.quality[i].clone()
        # In real mode availability derives from valid files/lab observations.
        if not USE_SYNTHETIC_DEMO:
            q[0] = q[0] * float(self.ecg[i].abs().sum() > 0)
            q[1] = q[1] * float(self.image[i].abs().sum() > 0)
        avail = torch.tensor([q[0] > 0, q[1] > 0, q[2] > 0], dtype=torch.bool)
        # Ensure lab modality is considered present if there are any lab features,
        # while its reliability still reflects completeness.
        return {"ecg": self.ecg[i], "image": self.image[i], "lab": self.lab[i],
                "labmask": self.labmask[i], "quality": q, "avail": avail, "y": self.y[i]}

train_ds = FMECAFDataset(Xtr_ecg, Xtr_img, Xtr_lab, Xtr_labmask, Qtr, y_train)
val_ds = FMECAFDataset(Xva_ecg, Xva_img, Xva_lab, Xva_labmask, Qva, y_val)
test_ds = FMECAFDataset(Xte_ecg, Xte_img, Xte_lab, Xte_labmask, Qte, y_test)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 10. RF-MambaFusion-style model
Compact modality encoders project ECG, radiograph and laboratory inputs into a common latent space. A learned quality estimator predicts modality reliability. Masked multi-head attention processes only available modalities, then reliability-weighted fusion feeds a binary classification head. This is a runnable lightweight substitute for the named pretrained encoders in the manuscript.

In [ ]:
class ECGEncoder(nn.Module):
    def __init__(self, d=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(12, 32, 9, stride=4, padding=4), nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32, 64, 7, stride=4, padding=3, groups=1), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.GELU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(96, d), nn.LayerNorm(d)
        )
    def forward(self, x): return self.net(x)

class ImageEncoder(nn.Module):
    def __init__(self, d=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2), nn.BatchNorm2d(16), nn.GELU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1, groups=1), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 96, 3, stride=2, padding=1), nn.GELU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(96, d), nn.LayerNorm(d)
        )
    def forward(self, x): return self.net(x)

class LabEncoder(nn.Module):
    def __init__(self, n_features, d=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features * 2, d), nn.GELU(),
                                 nn.LayerNorm(d), nn.Dropout(DROPOUT),
                                 nn.Linear(d, d), nn.LayerNorm(d))
    def forward(self, x, mask): return self.net(torch.cat([x, mask], dim=-1))

class ReliabilityAwareFusion(nn.Module):
    def __init__(self, n_lab, d=LATENT_DIM, heads=4):
        super().__init__()
        self.ecg_encoder = ECGEncoder(d)
        self.image_encoder = ImageEncoder(d)
        self.lab_encoder = LabEncoder(n_lab, d)
        self.quality_net = nn.Sequential(nn.Linear(3, 16), nn.GELU(), nn.Linear(16, 3), nn.Sigmoid())
        self.attn = nn.MultiheadAttention(d, heads, dropout=DROPOUT, batch_first=True)
        self.score = nn.Sequential(nn.Linear(d + 1, 32), nn.Tanh(), nn.Linear(32, 1))
        self.classifier = nn.Sequential(nn.Linear(d, 64), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(64, 1))
    def forward(self, batch, force_avail=None):
        z = torch.stack([
            self.ecg_encoder(batch["ecg"]),
            self.image_encoder(batch["image"]),
            self.lab_encoder(batch["lab"], batch["labmask"])
        ], dim=1)
        q = self.quality_net(batch["quality"])
        avail = batch["avail"] if force_avail is None else force_avail
        # Always keep at least one available branch, otherwise caller handles abstention.
        no_mod = ~avail.any(dim=1)
        if no_mod.any():
            avail = avail.clone()
            avail[no_mod, 2] = True
        contextual, _ = self.attn(z, z, z, key_padding_mask=~avail, need_weights=False)
        scores = self.score(torch.cat([contextual, q.unsqueeze(-1)], dim=-1)).squeeze(-1)
        scores = scores.masked_fill(~avail, -1e4)
        weights = torch.softmax(scores, dim=1) * q * avail.float()
        weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
        fused = torch.sum(contextual * weights.unsqueeze(-1), dim=1)
        logits = self.classifier(fused).squeeze(-1)
        return {"logits": logits, "prob": torch.sigmoid(logits), "reliability": q,
                "fusion_weights": weights, "no_modality": no_mod}

model = ReliabilityAwareFusion(n_lab=len(lab_cols)).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 11. Non-IID simulated federated client construction
Clients are created from training patients only. A Dirichlet label allocation approximates non-IID class proportions; each patient remains assigned to exactly one client. This is simulated federation, not a real multi-hospital deployment.

In [ ]:
def make_federated_clients(frame, n_clients=NUM_CLIENTS, alpha=0.5, seed=SEED):
    rng = np.random.default_rng(seed)
    client_indices = [[] for _ in range(n_clients)]
    for label in sorted(frame.cardiomegaly.unique()):
        idx = np.where(frame.cardiomegaly.to_numpy() == label)[0]
        rng.shuffle(idx)
        proportions = rng.dirichlet(np.full(n_clients, alpha))
        counts = rng.multinomial(len(idx), proportions)
        start = 0
        for c, count in enumerate(counts):
            client_indices[c].extend(idx[start:start+count].tolist())
            start += count
    return [np.array(sorted(ix), dtype=int) for ix in client_indices]

client_indices = make_federated_clients(train_df)
for c, ix in enumerate(client_indices):
    labels = train_df.iloc[ix].cardiomegaly.value_counts().to_dict() if len(ix) else {}
    print(f"Client {c}: n={len(ix)}, labels={labels}")
assert sum(len(x) for x in client_indices) == len(train_df)

## 12. Local training, modality dropout, and reliability-weighted aggregation
For a clear, runnable prototype, the server aggregates shared model parameters using encounter count and branch parameters using encounter count multiplied by mean branch reliability. Clients with no examples for a branch do not contribute to that branch. Local training uses weighted BCE and random modality dropout while retaining at least one modality.

In [ ]:
def subset_loader(dataset, indices, shuffle=True):
    from torch.utils.data import Subset
    return DataLoader(Subset(dataset, indices.tolist()), batch_size=BATCH_SIZE,
                      shuffle=shuffle and len(indices)>0, num_workers=0)

def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def local_train(global_state, dataset, indices, positive_weight, epochs=LOCAL_EPOCHS):
    local_model = ReliabilityAwareFusion(n_lab=len(lab_cols)).to(DEVICE)
    local_model.load_state_dict(global_state)
    local_model.train()
    opt = torch.optim.AdamW(local_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loader = subset_loader(dataset, indices, shuffle=True)
    if len(indices) == 0:
        return clone_state(local_model), 0.0, 0, np.zeros(3, dtype=float)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([positive_weight], device=DEVICE))
    losses = []
    for _ in range(epochs):
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            # Training-only modality dropout; never remove all currently available modalities.
            avail = b["avail"].clone()
            drop = torch.rand_like(avail.float()) < 0.15
            avail = avail & ~drop
            none = ~avail.any(dim=1)
            if none.any():
                avail[none, torch.randint(0, 3, (int(none.sum()),), device=DEVICE)] = True
            opt.zero_grad(set_to_none=True)
            out = local_model(b, force_avail=avail)
            loss = criterion(out["logits"], b["y"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
            opt.step()
            losses.append(float(loss.detach().cpu()))
    # Approximate branch quality summary from local training data.
    qsum = []
    local_model.eval()
    with torch.no_grad():
        for b in subset_loader(dataset, indices, shuffle=False):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            qsum.append(local_model.quality_net(b["quality"]).mean(dim=0).cpu().numpy())
    qmean = np.mean(qsum, axis=0) if qsum else np.zeros(3)
    return clone_state(local_model), float(np.mean(losses)) if losses else 0.0, len(indices), qmean

def aggregate_states(states, counts, reliabilities):
    # Weighted FedAvg for all parameters; modality-specific encoders receive reliability-aware weights.
    if not states:
        raise ValueError("No client updates to aggregate.")
    keys = states[0].keys()
    out = {}
    branch_prefix = {"ecg_encoder": 0, "image_encoder": 1, "lab_encoder": 2}
    for k in keys:
        if any(k.startswith(prefix) for prefix in branch_prefix):
            prefix = next(p for p in branch_prefix if k.startswith(p))
            j = branch_prefix[prefix]
            w = np.array([counts[i] * max(float(reliabilities[i][j]), 0.05) for i in range(len(states))], dtype=float)
        else:
            w = np.array(counts, dtype=float)
        if w.sum() <= 0: w = np.ones_like(w)
        w = w / w.sum()
        original_tensors = [states[i][k] for i in range(len(states))]
        # Integer buffers (e.g. num_batches_tracked) are copied from first state.
        if not original_tensors[0].is_floating_point():
            out[k] = original_tensors[0].clone()
        else:
            tensors = [t.float() for t in original_tensors]
            out[k] = sum(float(wi) * ti for wi, ti in zip(w, tensors))
    return out

def evaluate_loss(model, loader):
    model.eval(); losses=[]
    with torch.no_grad():
        for b in loader:
            b={k:v.to(DEVICE) for k,v in b.items()}
            logits=model(b)["logits"]
            losses.append(float(F.binary_cross_entropy_with_logits(logits,b["y"]).cpu()))
    return float(np.mean(losses)) if losses else float("nan")

pos = max(float((y_train == 0).sum()) / max((y_train == 1).sum(), 1), 1.0)
global_state = clone_state(model)
history = []
start_training = time.time()

for rnd in range(FED_ROUNDS):
    client_states, client_counts, client_reliabilities, local_losses = [], [], [], []
    for ix in client_indices:
        if len(ix) == 0:
            continue
        state, loss, count, qmean = local_train(global_state, train_ds, ix, pos)
        client_states.append(state); client_counts.append(count)
        client_reliabilities.append(qmean); local_losses.append(loss)
    global_state = aggregate_states(client_states, client_counts, client_reliabilities)
    model.load_state_dict(global_state)
    val_loss = evaluate_loss(model, val_loader)
    row = {"round": rnd+1, "mean_local_loss": float(np.mean(local_losses)), "val_loss": val_loss}
    history.append(row)
    print(row)

training_seconds = time.time() - start_training
hist_df = pd.DataFrame(history)
display(hist_df)

## 13. Training curves and validation calibration
Temperature scaling is fit only on validation logits. The test set is not used for model selection or calibration.

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(hist_df["round"], hist_df["mean_local_loss"], marker="o", label="Mean local training loss")
plt.plot(hist_df["round"], hist_df["val_loss"], marker="s", label="Validation BCE")
plt.xlabel("Federated round"); plt.ylabel("Loss"); plt.title("Federated training curves")
plt.legend(); plt.tight_layout(); plt.show()

def collect_outputs(model, loader):
    model.eval(); ys=[]; logits=[]; probs=[]; rel=[]; weights=[]
    with torch.no_grad():
        for b in loader:
            b={k:v.to(DEVICE) for k,v in b.items()}
            out=model(b)
            ys.extend(b["y"].cpu().numpy().tolist())
            logits.extend(out["logits"].cpu().numpy().tolist())
            probs.extend(out["prob"].cpu().numpy().tolist())
            rel.extend(out["reliability"].cpu().numpy().tolist())
            weights.extend(out["fusion_weights"].cpu().numpy().tolist())
    return np.asarray(ys), np.asarray(logits), np.asarray(probs), np.asarray(rel), np.asarray(weights)

yva, val_logits, val_probs, val_rel, val_weights = collect_outputs(model, val_loader)
yte, test_logits, test_probs_raw, test_rel, test_weights = collect_outputs(model, test_loader)

def fit_temperature(logits, labels):
    # Scalar temperature optimized by LBFGS on validation BCE.
    if len(np.unique(labels)) < 2:
        return 1.0
    lt = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(labels, dtype=torch.float32, device=DEVICE)
    log_t = torch.zeros((), requires_grad=True, device=DEVICE)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=50)
    def closure():
        opt.zero_grad()
        temp = torch.exp(log_t).clamp(0.05, 20.0)
        loss = F.binary_cross_entropy_with_logits(lt / temp, yt)
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.exp(log_t).clamp(0.05, 20.0).detach().cpu())

temperature = fit_temperature(val_logits, yva) if len(yva) else 1.0
test_probs = 1 / (1 + np.exp(-np.clip(test_logits / temperature, -40, 40)))
print(f"Validation-fitted temperature: {temperature:.4f}")

## 14. Test metrics and diagnostic plots
Metrics are computed from held-out test patients. AUROC/AUPR are reported only when both classes are present. No metric is hard-coded.

In [ ]:
def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan")
def safe_ap(y, p):
    return float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else float("nan")

def binary_metrics(y, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y.astype(int), pred, labels=[0,1]).ravel()
    specificity = tn / max(tn + fp, 1)
    sensitivity = tp / max(tp + fn, 1)
    return {
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall / Sensitivity": recall_score(y, pred, zero_division=0),
        "Specificity": specificity,
        "F1": f1_score(y, pred, zero_division=0),
        "MCC": matthews_corrcoef(y, pred) if len(np.unique(pred)) > 1 else 0.0,
        "AUROC": safe_auc(y, p),
        "AUPR / Average Precision": safe_ap(y, p),
        "Brier score": brier_score_loss(y, p),
        "Log loss": log_loss(y, np.clip(p, 1e-6, 1-1e-6), labels=[0,1]) if len(np.unique(y)) == 2 else float("nan"),
        "False-positive rate": fp / max(fp + tn, 1),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)
    }

metrics = binary_metrics(yte.astype(int), test_probs)
metrics_df = pd.DataFrame({"Metric": list(metrics.keys()), "Value": list(metrics.values())})
display(metrics_df)
print(classification_report(yte.astype(int), (test_probs >= 0.5).astype(int), labels=[0,1], zero_division=0))

fig, ax = plt.subplots(1, 2, figsize=(11,4))
cm = confusion_matrix(yte.astype(int), (test_probs >= .5).astype(int), labels=[0,1])
ax[0].imshow(cm); ax[0].set_title("Test confusion matrix"); ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")
for (i,j),v in np.ndenumerate(cm): ax[0].text(j,i,str(v),ha="center",va="center")
if len(np.unique(yte)) == 2:
    fpr, tpr, _ = roc_curve(yte, test_probs)
    ax[1].plot(fpr, tpr, label=f"AUROC={safe_auc(yte,test_probs):.3f}")
    ax[1].plot([0,1],[0,1],"--"); ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate")
    ax[1].set_title("ROC curve"); ax[1].legend()
plt.tight_layout(); plt.show()

## 15. Missing-modality robustness experiments
Each modality is removed from the test inputs at controlled rates, but the test labels remain unchanged. This is an evaluation-only stress test; it does not refit preprocessing or the model. If all three modalities are removed for a row, the model's fallback is not a valid clinical prediction and should be treated as insufficient input.

In [ ]:
def evaluate_with_missingness(model, dataset, rate, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    # Build a copy of dataset arrays and zero out selected modality tensors.
    ecg = dataset.ecg.numpy().copy()
    img = dataset.image.numpy().copy()
    lab = dataset.lab.numpy().copy()
    labmask = dataset.labmask.numpy().copy()
    quality = dataset.quality.numpy().copy()
    labels = dataset.y.numpy().copy()
    avail = np.ones((len(labels),3), dtype=bool)
    # Use dataset's original actual availability where available.
    for i in range(len(dataset)):
        item = dataset[i]
        avail[i] = item["avail"].numpy()
    for i in range(len(labels)):
        present = np.where(avail[i])[0]
        if len(present) > 1:
            for m in present:
                if rng.random() < rate and avail[i].sum() > 1:
                    avail[i,m] = False
        # Never manufacture an available modality if all are originally missing.
        if avail[i].sum() == 0:
            continue
        if not avail[i,0]: ecg[i] = 0; quality[i,0] = 0
        if not avail[i,1]: img[i] = 0; quality[i,1] = 0
        if not avail[i,2]: lab[i] = 0; labmask[i] = 0; quality[i,2] = 0
    ds = FMECAFDataset(ecg,img,lab,labmask,quality,labels)
    loader = DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False)
    # Override avail explicitly from simulated masks.
    model.eval(); ps=[]; ys=[]
    with torch.no_grad():
        offset=0
        for b in loader:
            n=len(b["y"])
            b={k:v.to(DEVICE) for k,v in b.items()}
            a=torch.tensor(avail[offset:offset+n],dtype=torch.bool,device=DEVICE)
            out=model(b, force_avail=a)
            ps.extend(out["prob"].cpu().numpy()); ys.extend(b["y"].cpu().numpy())
            offset += n
    p=1/(1+np.exp(-np.clip(np.log(np.clip(ps,1e-6,1-1e-6)/(1-np.clip(ps,1e-6,1-1e-6))) / temperature,-40,40)))
    result=binary_metrics(np.asarray(ys).astype(int),p)
    result["Missingness rate"] = rate
    return result

missing_rows=[]
for rate in MISSINGNESS_RATES:
    missing_rows.append(evaluate_with_missingness(model,test_ds,rate))
missing_df=pd.DataFrame(missing_rows)
display(missing_df[["Missingness rate","Accuracy","F1","AUROC","AUPR / Average Precision","False-positive rate"]])
plt.figure(figsize=(7,4))
plt.plot(missing_df["Missingness rate"],missing_df["F1"],marker="o",label="F1")
plt.plot(missing_df["Missingness rate"],missing_df["AUROC"],marker="s",label="AUROC")
plt.xlabel("Controlled modality removal probability"); plt.ylabel("Score"); plt.ylim(0,1.05)
plt.title("Performance under missing modalities"); plt.legend(); plt.tight_layout(); plt.show()

## 16. Reliability and fusion-weight analysis
These are learned model scores and should be interpreted as input-quality estimates, not probabilities of clinical correctness.

In [ ]:
if len(test_rel):
    rel_mean = test_rel.mean(axis=0)
    weight_mean = test_weights.mean(axis=0)
    names = ["ECG", "Radiograph", "Laboratory"]
    x = np.arange(3); w = 0.36
    plt.figure(figsize=(8,4))
    plt.bar(x-w/2, rel_mean, w, label="Estimated reliability")
    plt.bar(x+w/2, weight_mean, w, label="Mean fusion weight")
    plt.xticks(x,names); plt.ylim(0,1.05); plt.ylabel("Score"); plt.title("Modality reliability and contribution")
    plt.legend(); plt.tight_layout(); plt.show()

## 17. PC-EvidenceRAG: evidence corpus, retrieval, claim checks, and abstention
The evidence component below is deliberately conservative. It retrieves passages from a user-supplied corpus with provenance metadata and uses transparent lexical support heuristics. It does **not** claim to replace MedCPT/MedGemma or a validated medical entailment model. For real use, populate the corpus with permitted, curated clinical publications/guidelines and evaluate claim-level precision with expert annotations.

In [ ]:
DEFAULT_EVIDENCE = [
    {"title":"Cardiomegaly on chest radiographs", "date":"", "url":"https://pubmed.ncbi.nlm.nih.gov/",
     "text":"Cardiomegaly is an imaging finding that refers to an enlarged cardiac silhouette on chest radiography. Interpretation depends on image quality, projection, and clinical context."},
    {"title":"Clinical interpretation of ECG findings", "date":"", "url":"https://pubmed.ncbi.nlm.nih.gov/",
     "text":"Electrocardiography records cardiac electrical activity. ECG findings should be interpreted with symptoms, examination, and other investigations; a model prediction alone is not a diagnosis."},
    {"title":"Laboratory results and clinical context", "date":"", "url":"https://medlineplus.gov/lab-tests/",
     "text":"Laboratory measurements should be interpreted using the appropriate reference range, measurement units, patient context, and other clinical information. Missing values are not evidence of normal results."}
]

def load_evidence(path=EVIDENCE_CSV):
    if os.path.exists(path):
        ev = pd.read_csv(path).fillna("")
        if "text" not in ev.columns:
            raise ValueError("Evidence CSV must contain a 'text' column.")
        for c in ["title","url","date"]:
            if c not in ev.columns: ev[c] = ""
        return ev[["text","title","url","date"]].to_dict("records")
    return DEFAULT_EVIDENCE

evidence_docs = load_evidence()
vectorizer = TfidfVectorizer(stop_words="english", max_features=30000, ngram_range=(1,2))
evidence_matrix = vectorizer.fit_transform([d["text"] for d in evidence_docs])

def retrieve_evidence(query, top_k=5):
    qv = vectorizer.transform([query])
    scores = cosine_similarity(qv, evidence_matrix).ravel()
    order = np.argsort(scores)[::-1][:min(top_k, len(scores))]
    return [{"score":float(scores[i]), **evidence_docs[i]} for i in order]

def verify_claim_lexically(claim, retrieved, threshold=0.04):
    # Lightweight heuristic only; must be replaced/evaluated with medical NLI + expert review.
    cv = vectorizer.transform([claim])
    if not retrieved:
        return {"accepted":False,"support_score":0.0,"reason":"No evidence retrieved"}
    docs = [d["text"] for d in retrieved]
    dv = vectorizer.transform(docs)
    sims = cosine_similarity(cv, dv).ravel()
    best = float(sims.max()) if len(sims) else 0.0
    return {"accepted":best >= threshold,"support_score":best,
            "reason":"lexical similarity heuristic; not clinical entailment"}

def generate_conservative_explanation(pred_prob, reliability, observed_findings, question="Explain the model output",
                                      uncertainty_threshold=0.45, reliability_threshold=0.20):
    uncertainty = -(pred_prob*np.log(max(pred_prob,1e-8)) +
                    (1-pred_prob)*np.log(max(1-pred_prob,1e-8))) / np.log(2)
    rel = np.asarray(reliability).reshape(-1)
    query = f"{question}. Prediction: {'positive' if pred_prob >= .5 else 'negative'} cardiomegaly. " + str(observed_findings)
    retrieved = retrieve_evidence(query, top_k=5)
    if uncertainty > uncertainty_threshold or (len(rel) and float(rel.max()) < reliability_threshold) or not retrieved or max([d["score"] for d in retrieved], default=0) <= 0:
        return {"abstained":True, "prediction_probability":float(pred_prob),
                "uncertainty":float(uncertainty), "explanation":"Insufficient reliable input or supporting evidence to provide a specific explanation.",
                "citations":[], "retrieved":retrieved, "verified_claims":[]}
    label = "higher" if pred_prob >= .5 else "lower"
    claims = [
        f"The model assigned a {label} cardiomegaly probability ({pred_prob:.3f}).",
        "Cardiomegaly is an imaging finding and must be interpreted with image quality and clinical context."
    ]
    verified=[]
    for claim in claims:
        # The first claim is a faithful report of model output; the second needs corpus support.
        if claim.startswith("The model assigned"):
            verified.append({"claim":claim,"accepted":True,"support_score":1.0,"reason":"directly grounded in model output"})
        else:
            v=verify_claim_lexically(claim,retrieved)
            verified.append({"claim":claim,**v})
    accepted=[v["claim"] for v in verified if v["accepted"]]
    citations=[{"title":d["title"],"date":d["date"],"url":d["url"],"score":d["score"]} for d in retrieved if d["score"]>0]
    explanation=" ".join(accepted) if accepted else "The retrieved evidence did not sufficiently support a specific explanation; interpretation is withheld."
    return {"abstained":not bool(accepted), "prediction_probability":float(pred_prob),
            "uncertainty":float(uncertainty), "explanation":explanation,
            "citations":citations, "retrieved":retrieved, "verified_claims":verified}

print("Evidence documents:", len(evidence_docs))
print("Corpus source:", EVIDENCE_CSV if os.path.exists(EVIDENCE_CSV) else "small built-in demonstration corpus")

## 18. Generate a verified/abstained explanation for one test encounter
This prints a research demonstration only. Do not use it to make patient-care decisions.

In [ ]:
if len(test_probs):
    i = 0
    row = test_df.iloc[i]
    observed = {
        "encounter_id": str(row.get("encounter_id","")),
        "observed_lab_fraction": float(Mte_lab[i].mean()),
        "input_mode": "synthetic demonstration" if USE_SYNTHETIC_DEMO else "manifest-linked clinical data",
        "note": "No raw ECG interpretation is inferred by this text module."
    }
    explanation_result = generate_conservative_explanation(
        pred_prob=float(test_probs[i]), reliability=test_rel[i],
        observed_findings=observed,
        question="Provide a cautious explanation of the cardiomegaly classifier output"
    )
    print(json.dumps(explanation_result, indent=2))

## 19. Efficiency, communication volume, model size, and run summary
Communication volume is an estimate of serialized parameter payload, not a measured network transfer. GPU memory is reported only when CUDA is available. Detection time and privacy budget are not applicable unless a defined event-onset/drift protocol and a differential privacy mechanism are implemented.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
model_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
# Approximate model upload payload for each client each round (float32 parameter state).
state_bytes = sum(v.numel() * v.element_size() for v in model.state_dict().values())
estimated_client_upload_bytes = state_bytes * NUM_CLIENTS * FED_ROUNDS
summary = {
    "device": str(DEVICE),
    "train_encounters": int(len(train_df)),
    "validation_encounters": int(len(val_df)),
    "test_encounters": int(len(test_df)),
    "train_patients": int(train_df.patient_id.nunique()),
    "validation_patients": int(val_df.patient_id.nunique()),
    "test_patients": int(test_df.patient_id.nunique()),
    "num_clients": int(NUM_CLIENTS),
    "federated_rounds": int(FED_ROUNDS),
    "trainable_parameters": int(n_params),
    "parameter_bytes": int(model_bytes),
    "estimated_upload_bytes_all_rounds": int(estimated_client_upload_bytes),
    "training_seconds": float(training_seconds),
    "mean_test_inference_ms_per_encounter": None,
    "privacy_budget_epsilon": None,
    "note": "No differential privacy is enabled; privacy budget is therefore not reported."
}
# Inference latency estimate
if len(test_ds):
    model.eval()
    sample = next(iter(test_loader))
    sample = {k:v.to(DEVICE) for k,v in sample.items()}
    with torch.no_grad():
        for _ in range(2): model(sample)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0=time.time()
        reps=10
        for _ in range(reps): model(sample)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        elapsed=(time.time()-t0)/reps
    summary["mean_test_inference_ms_per_encounter"] = float(elapsed*1000/max(len(sample["y"]),1))
if DEVICE.type == "cuda":
    summary["peak_gpu_memory_mb"] = float(torch.cuda.max_memory_allocated()/1024**2)
print(json.dumps(summary, indent=2))

## 20. Save outputs (optional)
No model checkpoint is saved by default. The following cell writes only evaluation tables and training history to the current working directory. Disable it if running in a read-only location.

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    out_dir = Path("fmecaf_results"); out_dir.mkdir(exist_ok=True)
    hist_df.to_csv(out_dir/"federated_training_history.csv", index=False)
    metrics_df.to_csv(out_dir/"test_metrics.csv", index=False)
    missing_df.to_csv(out_dir/"missing_modality_metrics.csv", index=False)
    with open(out_dir/"run_summary.json","w") as f: json.dump(summary,f,indent=2)
    print("Saved tables to:", out_dir.resolve())
else:
    print("SAVE_RESULTS=False; no files written.")

## 21. Checklist before reporting research results
- [ ] Use the official Symile-MIMIC v1.0.0 release and retain its released patient-disjoint splits where applicable.
- [ ] Document cohort screening, label policy, image/ECG linkage, and exclusions.
- [ ] Verify all modalities correspond to the same patient/encounter and are available at prediction time.
- [ ] Replace compact prototype encoders with the exact Mamba-2, DINOv3 and tabular Transformer variants if claiming those architectures.
- [ ] Replace TF-IDF and lexical similarity with evaluated MedCPT retrieval/reranking and a suitable claim-support verifier; use MedGemma only where access and compute are available.
- [ ] Use multiple random seeds, patient-level bootstrap confidence intervals, ablations, and paired comparisons as proposed in the manuscript.
- [ ] Report only metrics produced by actual runs. Do not present synthetic-demo metrics as clinical performance.
- [ ] No differential privacy is implemented here; do not claim a privacy guarantee or epsilon value.
- [ ] Obtain appropriate institutional approvals and clinical review before any real-world use.